# pd.concat() — Unir DataFrames por posición

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/13_pandas_io/code/03_concat.ipynb)

`pd.concat()` es la herramienta cuando tienes DataFrames del **mismo tipo** y quieres pegarlos — ya sea apilando filas (axis=0) o columnas (axis=1). No sabe nada de claves ni relaciones: simplemente une por **posición**. El caso más común en la vida real: leer 12 archivos CSV (uno por mes) y juntarlos todos en uno.

In [ ]:
%pip install pandas==2.2.3 numpy==1.26.4

In [1]:
import pandas as pd
import numpy as np
from io import StringIO

## Sección 1: ¿Qué es concat?

`pd.concat()` une DataFrames **por posición**, no por significado. Hay dos direcciones:

**axis=0 (default) — apilar filas:**
```
df1:         df2:         resultado:
  A  B          A  B          A  B
0 1  2       0  5  6       0  1  2
1 3  4       1  7  8       1  3  4
                           0  5  6
                           1  7  8
```

**axis=1 — apilar columnas lado a lado:**
```
df1:         df2:         resultado:
  A  B          C  D          A  B  C  D
0 1  2       0  5  6       0  1  2  5  6
1 3  4       1  7  8       1  3  4  7  8
```

La regla es simple:
- `axis=0`: los DataFrames deben tener las **mismas columnas**
- `axis=1`: los DataFrames deben tener el **mismo índice**

## Sección 2: axis=0 — apilar filas

Caso típico: datos de ventas por mes, cada uno en su propio DataFrame.

In [2]:
df_enero = pd.DataFrame({
    "ciudad": ["CDMX", "GDL", "MTY"],
    "ventas": [100, 80, 60]
})

df_febrero = pd.DataFrame({
    "ciudad": ["CDMX", "GDL", "MTY"],
    "ventas": [120, 90, 70]
})

df_marzo = pd.DataFrame({
    "ciudad": ["CDMX", "GDL", "MTY"],
    "ventas": [110, 85, 75]
})

print("--- Enero ---")
print(df_enero)
print("\n--- Febrero ---")
print(df_febrero)

--- Enero ---
  ciudad  ventas
0   CDMX     100
1    GDL      80
2    MTY      60

--- Febrero ---
  ciudad  ventas
0   CDMX     120
1    GDL      90
2    MTY      70


### ignore_index=False (default) — conserva los índices originales

In [3]:
df_todo = pd.concat([df_enero, df_febrero, df_marzo])
print(df_todo)
print("\nÍndice:", df_todo.index.tolist())

  ciudad  ventas
0   CDMX     100
1    GDL      80
2    MTY      60
0   CDMX     120
1    GDL      90
2    MTY      70
0   CDMX     110
1    GDL      85
2    MTY      75

Índice: [0, 1, 2, 0, 1, 2, 0, 1, 2]


Nota que el índice tiene duplicados: 0, 1, 2, 0, 1, 2, 0, 1, 2. Eso puede causar problemas si intentas hacer `.loc[0]` — te regresa 3 filas.

### ignore_index=True — reinicia el índice de 0 a n

In [4]:
df_todo_limpio = pd.concat([df_enero, df_febrero, df_marzo], ignore_index=True)
print(df_todo_limpio)
print("\nÍndice:", df_todo_limpio.index.tolist())

  ciudad  ventas
0   CDMX     100
1    GDL      80
2    MTY      60
3   CDMX     120
4    GDL      90
5    MTY      70
6   CDMX     110
7    GDL      85
8    MTY      75

Índice: [0, 1, 2, 3, 4, 5, 6, 7, 8]


**¿Cuándo conservar el índice original?** Cuando el índice tiene significado — por ejemplo, si el índice es un `id_cliente` único. En ese caso, los duplicados en el índice son correctos (cada cliente puede aparecer en varios meses). Cuando el índice es solo un conteo sin significado, usa `ignore_index=True`.

💡 **Prueba esto:** Crea un `df_enero` donde el índice sea `[101, 102, 103]` (IDs de cliente) y observa qué pasa al hacer concat con y sin `ignore_index`.

## Sección 3: El problema de columnas disparejas con axis=0

¿Qué pasa si los DataFrames no tienen exactamente las mismas columnas?

In [5]:
df1 = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
df2 = pd.DataFrame({"a": [5, 6], "c": [7, 8]})  # columna "c" en lugar de "b"

print("df1:")
print(df1)
print("\ndf2:")
print(df2)

df1:
   a  b
0  1  3
1  2  4

df2:
   a  c
0  5  7
1  6  8


In [6]:
# join="outer" es el default — incluye todas las columnas, NaN donde no hay datos
result_outer = pd.concat([df1, df2], ignore_index=True)
print("join='outer' (default):")
print(result_outer)

join='outer' (default):
   a    b    c
0  1  3.0  NaN
1  2  4.0  NaN
2  5  NaN  7.0
3  6  NaN  8.0


In [8]:
# join="inner" — solo las columnas que existen en TODOS los DataFrames
result_inner = pd.concat([df1, df2], ignore_index=True, join="inner")
print("join='inner':")
print(result_inner)

join='inner':
   a
0  1
1  2
2  5
3  6


- `join='outer'` (default): toma la **unión** de columnas. Donde falta un valor, pone NaN.
- `join='inner'`: toma la **intersección** de columnas. Solo las que existen en todos los DataFrames.

💡 **Prueba esto:** Usa `join='inner'` cuando quieras un esquema estricto — si una fuente de datos tiene columnas extra o faltantes, el inner te avisa (resultado vacío o incompleto) en lugar de llenar silenciosamente con NaN.

## Sección 4: keys — crear un índice jerárquico

¿Cómo saber de qué chunk viene cada fila después del concat? Con `keys`.

In [9]:
df_con_keys = pd.concat(
    [df_enero, df_febrero, df_marzo],
    keys=["enero", "febrero", "marzo"]
)
print(df_con_keys)
print("\nTipo de índice:", type(df_con_keys.index))

          ciudad  ventas
enero   0   CDMX     100
        1    GDL      80
        2    MTY      60
febrero 0   CDMX     120
        1    GDL      90
        2    MTY      70
marzo   0   CDMX     110
        1    GDL      85
        2    MTY      75

Tipo de índice: <class 'pandas.core.indexes.multi.MultiIndex'>


In [10]:
# Acceder a un mes específico con .loc
print("Solo febrero:")
print(df_con_keys.loc["febrero"])

Solo febrero:
  ciudad  ventas
0   CDMX     120
1    GDL      90
2    MTY      70


In [11]:
# Iterar por grupos usando el nivel exterior del MultiIndex
for mes, grupo in df_con_keys.groupby(level=0):
    print(f"\nMes: {mes}")
    print(f"  Total ventas: {grupo['ventas'].sum()}")


Mes: enero
  Total ventas: 240

Mes: febrero
  Total ventas: 280

Mes: marzo
  Total ventas: 270


`keys` crea un `MultiIndex` donde el nivel exterior es la etiqueta que le diste a cada chunk. Útil cuando necesitas saber el origen de cada fila después del concat.

💡 **Prueba esto:** Haz `df_con_keys.reset_index(level=0)` para convertir el nivel exterior del MultiIndex en una columna regular llamada `level_0`. Así puedes tener la etiqueta del chunk como columna normal.

## Sección 5: axis=1 — apilar columnas

Cuando tienes información del mismo conjunto de entidades pero en DataFrames separados.

In [12]:
df_demograficos = pd.DataFrame({
    "nombre": ["Ana", "Luis", "María"],
    "edad": [28, 35, 42]
}, index=[101, 102, 103])

df_financieros = pd.DataFrame({
    "salario": [45000, 62000, 55000],
    "deuda": [5000, 0, 12000]
}, index=[101, 102, 103])

resultado = pd.concat([df_demograficos, df_financieros], axis=1)
print(resultado)

    nombre  edad  salario  deuda
101    Ana    28    45000   5000
102   Luis    35    62000      0
103  María    42    55000  12000


Funciona perfectamente porque los dos DataFrames tienen el **mismo índice** (101, 102, 103). Ahora veamos qué pasa cuando los índices no coinciden del todo:

In [14]:
df_demograficos_v2 = pd.DataFrame({
    "nombre": ["Ana", "Luis", "María"],
    "edad": [28, 35, 42]
}, index=[101, 102, 103])

# df_financieros_v2 tiene al cliente 104 pero no tiene al 103
df_financieros_v2 = pd.DataFrame({
    "salario": [45000, 62000, 70000],
    "deuda": [5000, 0, 3000]
}, index=[101, 102, 104])

resultado_v2 = pd.concat([df_demograficos_v2, df_financieros_v2], axis=1)
print(resultado_v2)

    nombre  edad  salario   deuda
101    Ana  28.0  45000.0  5000.0
102   Luis  35.0  62000.0     0.0
103  María  42.0      NaN     NaN
104    NaN   NaN  70000.0  3000.0


Cuando los índices no coinciden completamente, concat rellena con NaN — igual que un outer join. El cliente 103 no tiene datos financieros, y el cliente 104 no tiene datos demográficos.

**vs merge en índice:** `df.merge(df2, left_index=True, right_index=True)` es equivalente cuando los índices coinciden. Pero con merge puedes controlar el tipo de join (inner, left, right, outer) explícitamente. Con concat en axis=1, siempre obtienes un outer join por default.

## Sección 6: concat vs merge — la diferencia clave

| | `concat` | `merge` |
|---|---|---|
| Une por | Posición (filas o columnas) | Valores compartidos (claves) |
| Cuándo usarlo | Datos del mismo tipo/esquema | Datos relacionados por una clave |
| Equivalente SQL | `UNION ALL` (axis=0) | `JOIN` |
| Ejemplo | 12 CSVs de ventas mensuales | Tabla de clientes + tabla de pedidos |

La pregunta es: ¿estás **apilando** datos del mismo tipo, o **relacionando** tablas por una clave compartida?
- Apilar → `concat`
- Relacionar → `merge`

## Sección 7: Caso real — leer múltiples archivos CSV

Este es el patrón más común de `concat` en la vida real. Tienes un directorio con archivos CSV (uno por mes, por región, por experimento) y quieres unirlos todos.

In [15]:
# Simulamos múltiples archivos CSV con StringIO
# En la vida real serían: pd.read_csv("enero.csv"), o mejor:
# archivos = pathlib.Path("datos/").glob("*.csv")

csv_enero = StringIO("col_a,col_b,mes\n1,2,enero\n3,4,enero")
csv_febrero = StringIO("col_a,col_b,mes\n5,6,febrero\n7,8,febrero")
csv_marzo = StringIO("col_a,col_b,mes\n9,10,marzo\n11,12,marzo")

archivos = [csv_enero, csv_febrero, csv_marzo]

# El patrón: list comprehension + concat
dfs = [pd.read_csv(f) for f in archivos]
df_total = pd.concat(dfs, ignore_index=True)

print(df_total)

   col_a  col_b      mes
0      1      2    enero
1      3      4    enero
2      5      6  febrero
3      7      8  febrero
4      9     10    marzo
5     11     12    marzo


In [16]:
# Con pathlib en la vida real se vería así:
# import pathlib
#
# directorio = pathlib.Path("datos/ventas/")
# archivos = sorted(directorio.glob("*.csv"))  # sorted para orden consistente
#
# dfs = [pd.read_csv(f) for f in archivos]
# df_total = pd.concat(dfs, ignore_index=True)
#
# print(f"Total de filas: {len(df_total)}")
# print(f"Archivos leídos: {len(dfs)}")

print("Patrón: list comprehension + pd.concat(dfs, ignore_index=True)")
print(f"Total de filas en df_total: {len(df_total)}")
print(f"DataFrames concatenados: {len(dfs)}")

Patrón: list comprehension + pd.concat(dfs, ignore_index=True)
Total de filas en df_total: 6
DataFrames concatenados: 3


El patrón `[pd.read_csv(f) for f in archivos]` + `pd.concat(dfs, ignore_index=True)` es la forma idiomática de hacer esto en pandas. Evita el anti-patrón de hacer concat en un loop (que crea un nuevo DataFrame en cada iteración):

```python
# MAL — lento en archivos grandes:
df_total = pd.DataFrame()
for f in archivos:
    df_total = pd.concat([df_total, pd.read_csv(f)])  # copia en cada iteración

# BIEN — una sola llamada a concat:
dfs = [pd.read_csv(f) for f in archivos]
df_total = pd.concat(dfs, ignore_index=True)
```

## Sección 8: Ejercicio

Tienes datos de ventas de 4 ciudades durante 3 meses. Los datos están en DataFrames separados. Tu tarea:

1. Concatena todos los DataFrames en uno solo usando `keys` para identificar el mes
2. Calcula el **total de ventas por ciudad** (suma de los 3 meses)
3. Encuentra **qué mes tuvo las ventas totales más altas** (sumando todas las ciudades ese mes)

In [20]:
# Datos de partida — no modifiques esta celda
datos = {
    "enero": pd.DataFrame({
        "ciudad": ["CDMX", "GDL", "MTY", "PUE"],
        "ventas": [150, 90, 80, 45]
    }),
    "febrero": pd.DataFrame({
        "ciudad": ["CDMX", "GDL", "MTY", "PUE"],
        "ventas": [175, 110, 95, 60]
    }),
    "marzo": pd.DataFrame({
        "ciudad": ["CDMX", "GDL", "MTY", "PUE"],
        "ventas": [160, 100, 88, 52]
    })
}

print("Datos de enero:")
print(datos["enero"])

Datos de enero:
  ciudad  ventas
0   CDMX     150
1    GDL      90
2    MTY      80
3    PUE      45


In [34]:
# Tu solución aquí

# Paso 1: Concatena los DataFrames usando keys=["enero", "febrero", "marzo"]
df_ventas = pd.concat(
    list(datos.values()),
    keys=list(datos.keys())
)
print(f"DataFrame completo:")
print(df_ventas)

# Paso 2: Total de ventas por ciudad
# (pista: agrupa por la columna "ciudad")
ventas_por_ciudad = df_ventas.groupby("ciudad")["ventas"].sum().sort_values(ascending=False)
print(f"\nVentas por ciudad:")
print(ventas_por_ciudad)

# Paso 3: ¿Qué mes tuvo ventas totales más altas?
# (pista: agrupa por el nivel 0 del MultiIndex)
ventas_por_mes = df_ventas.groupby(level=0)["ventas"].sum()
mes_top = ventas_por_mes.idxmax()
print(f"\nMes con mayores ventas: {mes_top} ({ventas_por_mes[mes_top]} unidades)")

DataFrame completo:
          ciudad  ventas
enero   0   CDMX     150
        1    GDL      90
        2    MTY      80
        3    PUE      45
febrero 0   CDMX     175
        1    GDL     110
        2    MTY      95
        3    PUE      60
marzo   0   CDMX     160
        1    GDL     100
        2    MTY      88
        3    PUE      52

Ventas por ciudad:
ciudad
CDMX    485
GDL     300
MTY     263
PUE     157
Name: ventas, dtype: int64

Mes con mayores ventas: febrero (440 unidades)


In [ ]:
# Solución (descomenta para ver)

# # Paso 1
# df_ventas = pd.concat(
#     list(datos.values()),
#     keys=list(datos.keys())
# )
# print("DataFrame completo:")
# print(df_ventas)
#
# # Paso 2
# ventas_por_ciudad = df_ventas.groupby("ciudad")["ventas"].sum().sort_values(ascending=False)
# print("\nVentas totales por ciudad:")
# print(ventas_por_ciudad)
#
# # Paso 3
# ventas_por_mes = df_ventas.groupby(level=0)["ventas"].sum()
# mes_top = ventas_por_mes.idxmax()
# print(f"\nMes con mayores ventas: {mes_top} ({ventas_por_mes[mes_top]} unidades)")